# Notebook 09 — Streamlit frontend

This notebook validates and launches the CX Assist user interface. Keep Notebook 08 running because Streamlit communicates with its FastAPI endpoints.

## 1. Prerequisites

Keep these files in the same folder:

- `08_fastapi_backend.ipynb`
- `09_streamlit_frontend.ipynb`
- `streamlit_app.py`

Notebook 08 must show `LIVE API: READY`. Add `requests` to `requirement.txt` if it is not already installed.

In [1]:
from __future__ import annotations

import ast
import json
import os
import socket
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

APP_PATH = Path('streamlit_app.py')
API_URL = os.getenv('CX_ASSIST_API_URL', 'http://127.0.0.1:8000').rstrip('/')
STREAMLIT_HOST = '127.0.0.1'
STREAMLIT_PORT = 8501

if not APP_PATH.exists():
    raise FileNotFoundError('Place streamlit_app.py in the same folder as Notebook 09.')
print('Frontend file:', APP_PATH.resolve())
print('Backend URL:', API_URL)

Frontend file: C:\Users\Administrator\Documents\CX Assist\notebooks\streamlit_app.py
Backend URL: http://127.0.0.1:8000


## 2. Validate dependencies and frontend syntax

In [2]:
try:
    import pandas
    import requests
    import streamlit
except ImportError as exc:
    raise RuntimeError('Install streamlit, requests, and pandas, then restart the kernel.') from exc

app_source = APP_PATH.read_text(encoding='utf-8')
ast.parse(app_source)
print('STREAMLIT APP SYNTAX: PASSED ✅')

STREAMLIT APP SYNTAX: PASSED ✅


## 3. Confirm that Notebook 08 is still running

In [3]:
try:
    with urllib.request.urlopen(f'{API_URL}/health', timeout=10) as response:
        api_health = json.loads(response.read().decode('utf-8'))
except Exception as exc:
    raise RuntimeError('FastAPI is not reachable. Return to Notebook 08 and start its server cell.') from exc

print('Backend status:', api_health)
if api_health.get('status') != 'healthy':
    raise RuntimeError(f'Backend is not healthy: {api_health}')
print('BACKEND CONNECTION: PASSED ✅')

Backend status: {'status': 'healthy', 'provider': 'groq', 'model': 'openai/gpt-oss-120b', 'case_count': 5}
BACKEND CONNECTION: PASSED ✅


## 4. Start Streamlit

Run this cell once. It starts Streamlit as a separate local process.

In [4]:
def port_is_open(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.25)
        return sock.connect_ex((host, port)) == 0

if port_is_open(STREAMLIT_HOST, STREAMLIT_PORT):
    print(f'Port {STREAMLIT_PORT} is already in use. If CX Assist is open, use the existing server.')
else:
    streamlit_log = open('streamlit_app.log', 'w', encoding='utf-8')
    app_environment = os.environ.copy()
    app_environment['CX_ASSIST_API_URL'] = API_URL
    streamlit_process = subprocess.Popen(
        [
            sys.executable, '-m', 'streamlit', 'run', str(APP_PATH),
            '--server.address', STREAMLIT_HOST,
            '--server.port', str(STREAMLIT_PORT),
            '--server.headless', 'true',
            '--browser.gatherUsageStats', 'false',
        ],
        stdout=streamlit_log, stderr=subprocess.STDOUT, env=app_environment,
    )
    for _ in range(100):
        if streamlit_process.poll() is not None:
            streamlit_log.flush()
            raise RuntimeError(Path('streamlit_app.log').read_text(encoding='utf-8'))
        if port_is_open(STREAMLIT_HOST, STREAMLIT_PORT):
            break
        time.sleep(0.1)
    if not port_is_open(STREAMLIT_HOST, STREAMLIT_PORT):
        raise RuntimeError('Streamlit did not start within 10 seconds. Check streamlit_app.log.')
    print(f'Streamlit running with PID {streamlit_process.pid}')

Streamlit running with PID 1340


## 5. Confirm the live frontend

In [5]:
frontend_url = f'http://{STREAMLIT_HOST}:{STREAMLIT_PORT}'
with urllib.request.urlopen(frontend_url, timeout=15) as response:
    frontend_status = response.status
print('Frontend HTTP status:', frontend_status)
print('Open:', frontend_url)
print('CX ASSIST FRONTEND: READY ✅')

Frontend HTTP status: 200
Open: http://127.0.0.1:8501
CX ASSIST FRONTEND: READY ✅


## 6. Demo walkthrough

1. Select `CASE-3022` or another available case.
2. Show Case 360 before starting the investigation.
3. Start the investigation and explain the evidence/policy grounding.
4. Review the recommendation and customer-response draft.
5. Approve, edit, reject, or escalate.
6. Show the final response and newly created audit-history entry.

The interface never sends a customer response automatically; it only records the human-reviewed outcome.

## 7. Stop Streamlit when finished

In [ ]:
# Run only when the demo is finished.
# if 'streamlit_process' in globals() and streamlit_process.poll() is None:
#     streamlit_process.terminate()
#     streamlit_process.wait(timeout=10)
#     streamlit_log.close()
#     print('Streamlit stopped.')